In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import ccf
from statsmodels.graphics.tsaplots import plot_acf
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
IN = '../data/raw'
OUT = '../data/python_master'

df = pd.read_csv(f"{IN}/planning_applications/PS2_data_-_open_data_table__202512_.csv",
                  encoding="cp1252", skiprows=2)

# list_cols = df.columns.to_list()
# for col in list_cols:
#     if "Total granted" in col:
#         print(col)

# Columns we need: Total granted; major total (all), Total granted; minor total(all)

df['Total_Granted'] = df["Total granted; major total (all)"]

qs = df['Quarter'].str.replace(" ", "")
df['Quarter'] = pd.PeriodIndex(qs, freq='Q')

# Groupping by quarter then summing for England total- may revisit file later on for regional analysis
df_eng = df.groupby('Quarter')['Total_Granted'].sum().reset_index()

df_eng.to_csv(f"{OUT}/planning/plan_granted.csv")

In [ ]:
df_eng.set_index('Quarter')['Total_Granted'].plot()

In [ ]:
df_starts = pd.read_csv(f"{OUT}/england_master.csv")

df_starts = df_starts.iloc[:,:2]

df_starts.columns = ['Quarter', 'Starts']

df_starts['Quarter'] = pd.PeriodIndex(df_starts['Quarter'], freq='Q')


df = pd.merge(df_starts, df_eng, on='Quarter')

df

# Starts was found to be trend stationary I(0), but we choose to difference so we dont compare level-diff
#  We convert total granted to I(0) by differencing adjusting for seasonality
df['dTotal_Granted_SA'] = df['Total_Granted'].diff(4)
df['dStarts_SA'] = df['Starts'].diff(4)
df = df.dropna()

# Finding cross correlations across lags
result = ccf(df['dStarts_SA'], df['dTotal_Granted_SA'])

n_lags = 12

n = len(df)
conf = 1.96 / np.sqrt(n)

plt.stem(range(n_lags), result[:n_lags])
plt.axhline(0, color='black', linewidth=0.5)
plt.axhline(conf, color='gray', linestyle='--')
plt.axhline(-conf, color='gray', linestyle='--')
plt.xlabel('Lag (quarters)')
plt.ylabel('Cross-correlation')
plt.axhline(0, color='black', linewidth=0.5)
plt.title('Granted (lagged) vs Starts')
plt.show()

# Co-movement with 0-1 quarter - we choose 1

# Checking ACF of Total Granted
plot_acf(df['dTotal_Granted_SA'].dropna(), lags=12)
df.head()

In [ ]:
df['dStarts_SA_lag1'] = df['dStarts_SA'].shift(1)

# AR(1) 
X_a = sm.add_constant(df[['dStarts_SA_lag1']])
model_a = sm.OLS(df['dStarts_SA'], X_a, missing='drop').fit(cov_type='HAC', cov_kwds={'maxlags':4})

# AR(1) + permissions lag
df['dGranted_SA_lag1'] = df['dTotal_Granted_SA'].shift(1)
X_b = sm.add_constant(df[['dStarts_SA_lag1', 'dGranted_SA_lag1']])
model_b = sm.OLS(df['dStarts_SA'], X_b, missing='drop').fit(cov_type='HAC', cov_kwds={'maxlags':4})

print(model_a.summary())
print(model_b.summary())